# Encoding Geographical Risk from Global Sanctions Datasets

## A Structured Open Source Intelligence Tool to support Regulatory and AML/CFT Compliance
---

**Author**: Francesco Silvano, MEng in Engineering Management  

**Instructors**: Prof. Marco Maggini, Prof. Federico Becattini

**Course**: Business Intelligence, University of Siena 

**Publication Date**: June 2026

---

## Introduction & Context

Sanctions designation lists are maintained by dozens of jurisdictions — OFAC, the EU, the UN Security Council, and others — each with independent update cycles, entity formats, and coverage priorities. Cross-referencing these lists at scale is operationally expensive for compliance teams, and the absence of a unified analytical layer makes systematic pattern detection across regimes difficult for researchers and policy analysts alike.

This project builds a basic sanctions intelligence pipeline on top of [OpenSanctions](https://www.opensanctions.org/), a consolidated open-source repository of global sanctions and politically exposed persons (PEPs) data. The pipeline ingests and combines multiple datasets, enriches records through complementary data joins — adding sanctioned maritime vessels and financial securities — and validates data quality before analysis is run.

Three analytical techniques are applied: a **market-basket model** using the A-Priori algorithm to identify frequent co-occurrence patterns in entity attributes; a **recommendation system** based on UV decomposition via SGD to detect cross-list coverage gaps; and a **binary classifier** to assess whether a PEP profile is statistically consistent with sanctions designation. Findings are exposed through an interactive Quarto dashboard rendering this notebook.

The intended use cases are AML/CFT compliance screening, defense-sector entity monitoring, and geopolitical risk analysis.

---

### Methodology

The built pipeline ingests and combines multiple datasets, cleans data and applies targeted preprocessing to ensure data is in a usable format and that any inconsistencies or missing values are addressed; an enrichment layer includes sanctioned maritime vessels and financial securities through complementary data joins - providing a holistic view of the activities carried out by sanctioned entities.

- In _Part I_ the datasets are explored to understand entity distributions, data quality, and country coverage across different sanction types;
- in _Part II_ three analytical layers are built: a **market-basket model** using the A-Priori algorithm identifies frequent co-occurrence patterns in entity attributes; **Jaccard Similarity** is applied to infer potential links between vessel records and sanctioned entities, and to rank non-sanctioned PEPs by profile similarity to designated ones; a **recommendation system** based on UV decomposition via SGD surfaces cross-list coverage gaps — flagging entities likely under-designated relative to their attribute profile;
- In _Part III_, a **simple binary classifier ML model** using linear and quadratic discriminant analysis is proposed to assess whether a PEP profile is statistically consistent with sanctions designation. A large-scale stochastic gradient descent (SGD) classifier is added as a higher-dimensional benchmark to attempt to predict risk levels based on geographical and entity-specific features, providing a probabilistic assessment of potential sanctions evasion.

#### A few assumptions on data quality to be aware of

- The analysis is based on the assumption that the datasets used are comprehensive and accurate representations of the entities and relationships involved in sanctions. While OpenSanctions is an authorative source, and their updates are frequent, the available data may not capture all information in real time, and there may be gaps or inaccuracies.

- Data is distributed in the **_FollowTheMoney (FTM)_** format, which is a specific structure for representing entities and their relationships. This format is in JSON, which requires significant preprocessing to be transformed into a structured format suitable for analysis. For the purpose of this project, a simpler version of the data in CSV format is used, which may not capture all the nuances and details present in the original JSON data. A pratical example: sanctions designation wouldn’t be in the PEPs collection because those sources don’t contain the designations texts. 

- The data originates from a variety of sources and therefore varies in terms of detail and precision (e.g., many listings of political office holders do not include their exact birth dates or full given names). This represents a significant challenge for data quality, as it can lead to difficulties in entity resolution and risk assessment. For instance, the lack of precise identifiers may result in false positives or negatives when screening for sanctioned entities, which can have serious implications for compliance efforts.

- The cut-off date for the data is set to 10/06/2026, and any events or changes that occurred after this date are not included in the analysis. This means that the findings may not reflect the most current state of sanctions and related activities.

---

### Pipeline Architecture

```Data ingestion
├── consolidated_sanctions.csv ──► df_consolidated
├── peps.csv ────────────────────► df_peps  ──► df_peps_work      [dedup: max non-null per id]
├── maritime.csv ────────────────► df_maritime
└── securities.csv ──────────────► df_securities                  [EDA only]

Part I — Preprocessing
└── df_consolidated  ×  df_peps_work
    ├── df_sanctioned_people          [schema == 'Person']
    │   ├── df_sanctioned_people_nopeps   [id ∉ df_peps_work]
    │   └── df_sanctioned_people_peps    [inner join on id]  ──► used in RQ4 and Part III
    └── country distribution charts  (4 datasets, countries split on ';' and exploded)

Part II — Analysis

├── RQ1  Jaccard Similarity
│   ├── in:     df_maritime['caption']  ×  df_consolidated['name']
│   ├── method: char trigrams → MinHash LSH (threshold=0.5, num_perm=64) → exact Jaccard re-check
│   └── out:    df_rq1  [vessel–entity candidate pairs, J ≥ 0.5]
│
├── RQ2  Frequent Itemsets  (A-Priori)
│   ├── in:     df_maritime
│   ├── method: baskets {country:XX, flag:YY, conduit:True}
│   │           → TransactionEncoder → apriori (min_support=0.03, max_len=3)
│   │           → association_rules (confidence ≥ 0.4)
│   └── out:    flag_rules  [lift > 1.2, filtered to flag/conduit consequents]
│
├── RQ3  Recommendation System  (UV Decomposition)
│   ├── in:     df_consolidated['dataset']
│   ├── method: entity × programme binary matrix M
│   │           → SGD factorisation  U (n × 10) · V (10 × p)  [η=0.01, λ=0.02, 5 epochs]
│   │           → scores = U @ V,  known entries masked to -∞
│   └── out:    top-5 missing programme recommendations per entity
│
└── RQ4  Collaborative Filtering  (Jaccard on attribute sets)
    ├── in:     df_peps_work  ×  df_sanctioned_people_peps
    ├── method: attribute sets {country:XX} → pairwise Jaccard (reference sample = 50)
    │           → mean similarity score per non-sanctioned PEP
    └── out:    non_sanctioned ranked by risk_score

Part III — Machine Learning
├── in:     df_peps_work → df_part3
│   ├── features:  n_aliases · has_birthdate · n_identifiers
│   └── label:     1 if id ∈ df_sanctioned_people_peps  else  0
│
├── LDA   [X_simple, 3 features]   linear boundary · equal covariance assumed
├── QDA   [X_simple, 3 features]   quadratic boundary · per-class covariance
└── SGD   [X_sgd,   33 features]   X_simple + top-30 country one-hot · modified_huber loss
    └── out: AUC comparison table  (LDA · QDA · SGD)
```

### Possible future applications
---

The intended use cases are compliance screening, defense-sector entity monitoring, and geopolitical risk analysis.

| # | Application | Description | Example |
| --- | --- | --- | --- |
| 1 | AML/CFT Counterparty Screening | Automated screening of counterparties, vendors, and transaction parties against the consolidated sanctions list. Fuzzy matching catches name variants and transliterations used to evade direct string matches — a common evasion technique. | A correspondent bank screens incoming wire transfers; the fuzzy matching layer flags "Rosneft Trading S.A." as a near-match to the designated "Rosneft Trading SA" on the EU FSF list, triggering an enhanced due diligence review before funds are released. |
| 2 | Sanctions Evasion Pattern Detection | Frequent itemset mining surfaces co-occurrence patterns in entity attributes — jurisdiction, schema type, cross-list membership — that characterise known evasion structures such as offshore shells, conduit flag states, and layered ownership chains. | The A-Priori algorithm identifies that entities sharing the profile {Company, country:AE, listed in OFAC} disproportionately lack an EU FSF designation. This cross-list coverage gap is consistent with UAE-incorporated intermediaries used to route sanctioned capital between Russian principals and European markets. |
| 3 | Geopolitical and Sectoral Risk Intelligence | Network centrality analysis maps which industries, jurisdictions, and corporate archetypes concentrate the most sanctioned connections — exposing emerging evasion corridors before they appear in formal designations. | A defense ministry analyst uses betweenness centrality scores to identify a cluster of Turkish shipping companies acting as structural brokers between sanctioned Russian oil producers and Asian buyers — a pattern not yet reflected in any active designation list. |
| 4 | Investigative Research and OSINT | An interactive dashboard for analysts, journalists, and NGOs to explore entity networks, trace ownership chains, and surface previously unknown connections between designated persons and ostensibly clean entities. The recommendation system surfaces candidate connections the user has not yet examined. | An investigative journalist queries the dashboard to trace the ownership chain of a vessel flagged under a conduit jurisdiction, linking it through three shell companies back to a sanctioned oligarch whose name does not appear in any public vessel registry. |

---

### Libraries and Imports

In [ ]:
# Data
import pandas as pd
import numpy as np
from scipy import stats

# Other tools
import json, re, hashlib, itertools, collections, math, warnings
from pathlib import Path

# Similarity
from datasketch import MinHash, MinHashLSH        # MinHash-based Jaccard LSH

# A-Priori Algorithm
from mlxtend.frequent_patterns import apriori, association_rules 
from mlxtend.preprocessing import TransactionEncoder

# LDA, QDA use module: scikit learn
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.inspection import DecisionBoundaryDisplay

# Vizualization
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

# Machine Learning
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, roc_auc_score, average_precision_score)
from sklearn.model_selection import StratifiedKFold
from scipy.sparse import lil_matrix, csr_matrix

DATA_PATH = "data/input"

### Collection and Update Process

The dataset coverage — i.e. when it was started and how frequently it is updated — is every six hours, and reflects the latest submissions and updates from the original sources. 

The dataset source is updated automatically; however, for the purposes of this project, a static snapshot of the data was taken on 10 June 2026, the date on which the final analysis was run, to ensure consistency across the analysis and avoid potential issues arising from changes to the data during the project timeline.

## Part I: Data Exploration and Preprocessing
---

### Cleaning the main datasets

The datasets required minimal pre-processing as they were sourced from a structured, actively maintained repository. Two issues were identified and addressed:

- Multi-value country fields storing semicolon-separated codes as a single string, which required splitting before any country-level analysis.
- Duplicate entries in the PEPs dataset across sources, resolved by retaining the most complete record per unique entity.

In particular, the PEPs dataset required a deduplication process due to duplicates present in different original data sources.

This process involved keeping the occurrence with the most complete information, i.e. the row with the fewest missing values across all columns. This approach ensured that the most informative record for each unique entity was retained while eliminating redundant entries that could have jeopardised the accuracy of our analysis.

### Describing the main datasets

#### DS1 Consolidated Sanctions

The _"Consolidated Sanctions"_ dataset is a distribution listing all the sanctioned entities designated by different countries and  international organisations. This can include military, trade and travel restrictions.

This collection combines all designated entities from the various sanctions lists included in OpenSanctions. It represents the broadest set of sanctioned people, companies and other assets that is designated globally.

In [ ]:
# Load the data for Consolidated Sanctions dataset
df_consolidated = pd.read_csv(f"{DATA_PATH}/consolidatedsanctions/targets.simple-1.csv", low_memory=False).drop(columns=['phones','emails']) # phones and emails are dropped for simplicity
df_consolidated['countries'] = df_consolidated['countries'].str.upper()

Some records may have multiple `countries` separated by a semicolon. This means that specific entry has links to two countries. For example, a string like "RU;CY" could have different meanings such as _"Russian oligarch with Cypriot residency"_ or _"Russian company with a Cypriot branch"_.

In [ ]:
df_consolidated['countries'].str.contains(';', na=False).sum()

The number of those entities is **4575**. If left as-is, each entity would appear zero times in the RU count and zero times in the CY count. It almost certainly falls below a 2% threshold as a combined string, so it lands in the "Others" category, so this category would be artificially inflated. This represents a systematic bias against multi-jurisdiction entities. 

Therefore multi-country entities are disproportionately the analytically interesting ones - dual-national oligarchs, shell companies registered across multiple jurisdictions, intermediaries with deliberate geographic complexity.

#### DS2 Politically Exposed Persons (PEPs) dataset

The _"Politically Exposed Persons (PEPs)"_ dataset is a collection of individuals who are or have been entrusted with prominent public roles.

This dataset is important for financial institutions and other entities that are required to conduct enhanced due diligence on PEPs to mitigate risks of corruption and money laundering. It includes information on the individual's name, position, country of origin, and any associated risks.

In [ ]:
# Load the data for the PEPs dataset
df_peps = pd.read_csv(f'{DATA_PATH}/peps/targets.simple-1.csv', low_memory=False) # low_memory=False to avoid dtype warnings

df_peps['countries'] = df_peps['countries'].str.upper()
df_peps

In [ ]:
df_peps.head()
n_peps = df_peps['id'].nunique()
n_duplicates = df_peps['name'].duplicated().sum()

print(f"Number of unique entities in PEPs dataset: {n_peps}")
print(f"Number of duplicates in PEPs dataset: {n_duplicates}")

df_duplicates = df_peps[df_peps['name'].duplicated(keep=False)].sort_values('name')
df_duplicates

In [ ]:
df_peps_work = df_peps.copy()
df_peps_work['non_null_count'] = df_peps_work.count(axis=1)

df_peps_work = df_peps_work.sort_values('non_null_count', ascending=False).drop_duplicates(subset='id', keep='first')

In [ ]:
df_peps_work

In [ ]:
df_sanctioned_people = df_consolidated[df_consolidated['schema'] == 'Person']
df_sanctioned_people

The `df_sanctioned_people` dataframe contains only the entities from the Consolidated Sanctions dataset that are classified as 'Person' under the 'schema' column. These are all the individuals that have been designated as sanctioned entities by different countries and international organisations.

#### Distribution of sanctioned individuals by country

##### Sanctioned individuals that are not PEPs

In [ ]:
df_sanctioned_people_nopeps = df_sanctioned_people[~df_sanctioned_people['id'].isin(df_peps_work['id'])]
df_sanctioned_people_nopeps

In [ ]:
country_counts_nopeps = df_sanctioned_people_nopeps['countries'].str.split(';').explode().str.strip().value_counts()

country_percentage_nopeps = country_counts_nopeps / country_counts_nopeps.sum()

main_countries_nopeps = country_counts_nopeps[country_percentage_nopeps >= 0.02]
other_countries_nopeps = country_counts_nopeps[country_percentage_nopeps < 0.02].sum()

plot_data = main_countries_nopeps.copy()
if other_countries_nopeps > 0:
    plot_data['Others'] = other_countries_nopeps

fig = plt.figure(figsize=(10, 6))
plot_data.plot(
    kind='pie',
    autopct='%1.1f%%', # autopct to show percentage on the pie chart
    startangle=90,
    pctdistance=0.8
)

plt.title('Distribution of sanctioned individuals by country')
plt.axis('equal')
plt.tight_layout()
plt.legend()
plt.show()

##### Sanctioned individuals that are also PEPs

Analyzing the distribution of the sanctioned PEPs by country allows us to understand how many politically exposed persons are sanctioned in each country, and to identify any patterns or trends in the distribution of sanctions across different countries. This can provide insights into the geopolitical and economic factors that may influence sanctions designation, and can inform risk assessment and compliance efforts for entities operating in those countries.

In [ ]:
df_sanctioned_people_peps = pd.merge(df_sanctioned_people, df_peps_work, on='id', how='inner', suffixes=('_consolidated', '_peps'))
df_sanctioned_people_peps

The resulting dataset is typically named "The Russian Government" by the OpenSanctions team for the presence of many high-profile Russian politicians and oligarchs who are both sanctioned and politically exposed. This subset can potentially be analyzed further to understand the overlap between the two datasets and to identify any unique characteristics of these entities, but it's out of project scope.

In [ ]:
country_counts_peps = df_sanctioned_people_peps['countries_peps'].str.split(';').explode().str.strip().value_counts()

country_percentage_peps = country_counts_peps / country_counts_peps.sum()

main_countries_peps = country_counts_peps[country_percentage_peps >= 0.02]
other_countries_peps = country_counts_peps[country_percentage_peps < 0.02].sum()

plot_data = main_countries_peps.copy()
if other_countries_peps > 0:
    plot_data['Others'] = other_countries_peps

fig = plt.figure(figsize=(10, 6))
plot_data.plot(
    kind='pie',
    autopct='%1.1f%%', # autopct to show percentage on the pie chart
    startangle=90,
    pctdistance=0.8
)

plt.title('Distribution of sanctioned PEPs by country')
plt.axis('equal')
plt.tight_layout()
plt.legend()
plt.show()

### Describing the Enrichment Layers

#### Distribution of sanctioned non-individual entities by country

In this section, we introduce two layers of enrichment. The first layer identifies _**Maritime Vessels**_ subject to sanctions and attempts to uncover hidden links to sanctioned entities. Vessels can be used to circumvent trade sanctions and are not always directly linked to sanctioned entities; they are sometimes linked to shell companies that mask the main entity, which may be an organisation, a company or, though this is rare, a high-profile individual. The second layer concerns sanctioned securities. By _**"Sanctioned Security"**_, we mean a tradable (though not necessarily traded) financial asset that holds monetary value. The specialised "Sanctioned Securities" dataset maps sanctioned companies to their tradable financial assets.

##### EL1 Sanctioned vessels

In [ ]:
df_maritime = pd.read_csv(f'{DATA_PATH}/maritime/maritime.csv', low_memory=False) # low_memory=False to avoid dtype warnings
df_maritime['countries'] = df_maritime['countries'].str.upper()
df_maritime['flag'] = df_maritime['flag'].str.upper()

In [ ]:
df_maritime

In [ ]:
country_counts = df_maritime['countries'].str.split(';').explode().str.strip().value_counts()

country_percentage = country_counts / country_counts.sum()

main_countries = country_counts[country_percentage >= 0.02]
other_countries = country_counts[country_percentage < 0.02].sum()

plot_data = main_countries.copy()
if other_countries > 0:
    plot_data['Others'] = other_countries

fig = plt.figure(figsize=(10, 6))
plot_data.plot(
    kind='pie',
    autopct='%1.1f%%', # autopct to show percentage on the pie chart
    startangle=90,
    pctdistance=0.8
)

plt.title('Distribution of sanctioned vessels by country')
plt.axis('equal')
plt.tight_layout()
plt.legend()
plt.show()

##### EL2 Sanctioned securities

In [ ]:
df_securities = pd.read_csv(f'{DATA_PATH}/securities/securities.csv', low_memory=False)
df_securities['countries'] = df_securities['countries'].str.upper()

In [ ]:
df_securities

The `lei` column contains the Legal Entity Identifier for each security, which is a unique identifier used to identify legal entities participating in financial transactions. The `countries` column indicates the country associated with each security, and the `name` column provides the name of the security. This dataset can be used to analyze the characteristics of sanctioned securities and their associated countries.

In [ ]:
country_counts = df_securities['countries'].str.split(';').explode().str.strip().value_counts()

country_percentage = country_counts / country_counts.sum()

main_countries = country_counts[country_percentage >= 0.02]
other_countries = country_counts[country_percentage < 0.02].sum()

plot_data = main_countries.copy()
if other_countries > 0:
    plot_data['Others'] = other_countries

fig = plt.figure(figsize=(10, 6))
plot_data.plot(
    kind='pie',
    autopct='%1.1f%%', # autopct to show percentage on the pie chart
    startangle=90,
    pctdistance=0.8
)

plt.title('Distribution of sanctioned securities by country')
plt.axis('equal')
plt.tight_layout()
plt.legend()
plt.show()

## Part II: Data Analysis and Modeling
---

In [ ]:
# This analytical layer employs a market-basket model using the A-Priori algorithm to surface frequent co-occurrence patterns in entity attributes. 

# The analytical layer is structured around three techniques: a **market-basket model** using the A-Priori algorithm to surface frequent co-occurrence patterns in entity attributes; a **recommendation system** based on UV decomposition via SGD to detect cross-list coverage gaps;

# and a basic **machine learning classifier** to assess whether a PEP profile is consistent with sanctions designation.

| # | Question | Use Case | Technology  | Inputs |
| --- | -------- | --------------------- | -------| ------ |
| RQ1 | Which maritime vessels are potentially linked to a sanctioned entity despite not sharing a common ID - i.e. were they renamed or re-registered to conceal ownership? | Sanctions evasion via vessel renaming | **Jaccard Similarity** on character n-grams of `caption` and `name` fields: compare `df_maritime['caption']` against `df_consolidated['name']`; pairs above threshold are candidate links | `df_maritime`, `df_consolidated` |
| RQ2 | Is there a statistically significant association between a vessel's country of sanctioned ownership and its registered flag state? | Maritime sanctions evasion; dark fleet detection | Frequent Itemsets (A-Priori): baskets = vessels; items = {country:XX, flag:YY}; derive rule {country:RU} → {flag:PA} and compute confidence and lift against the baseline flag distribution; conduit_jurisdictions variable is the starting point | `df_maritime`, `conduit_jurisdictions` |
| RQ3 | Given an entity's current dataset memberships, which additional sanctions programmes is their profile most consistent with — i.e., which programmes have they not yet been listed in but should be? | Cross-list gap detection; targeted investigation | Recommendation System: build a binary entity × dataset utility matrix from datasets column; apply UV decomposition (SGD) to compute latent risk factors; score unobserved (entity, programme) pairs | `df_consolidated`, `df_peps_work` |
| RQ4 | Which non-sanctioned PEPs are most similar to the entities in `df_sanctioned_people_peps`, and should therefore be prioritised for enhanced due diligence? | AML/KYC prioritisation | Recommendation System (user-user collaborative filtering): Jaccard similarity on attribute sets between PEP-only entities and sanctioned PEPs; rank PEP-only entities by their average similarity to their nearest sanctioned neighbours | `df_peps_work`, `df_sanctioned_people_peps` |

### RQ1: Which maritime vessels are potentially linked to a sanctioned entity despite not sharing a common ID, i.e. were they renamed or re-registered to conceal ownership?

We use **_Jaccard Similarity_** to measure lexical similarity on name strings (detecting renamed vessels); use character n-grams of `name` field: compare `df_maritime['caption']` against `df_consolidated['name']`; pairs above threshold are candidate links - datasets `df_maritime`, `df_consolidated`

In [ ]:
# Build n-gram sets
def ngrams(s, n=3):
    s = s.lower().strip()
    return set(s[i:i+n] for i in range(len(s) - n + 1))

vessel_names  = df_maritime[['id','caption']].dropna()
entity_names  = df_consolidated[['id','name']].dropna()

vessel_names['ngrams'] = vessel_names['caption'].apply(ngrams)
entity_names['ngrams'] = entity_names['name'].apply(ngrams)

# RQ1 name similarity uses only lowercase/strip normalization and raw trigram sets, which may be noisy for vessel renaming.

# Add stronger text normalization (punctuation removal, common vessel prefixes, transliteration handling) or use token-based fuzzy matching to reduce false positives.

In [ ]:
lsh = MinHashLSH(threshold=0.5, num_perm=64) # MinHashLSH is used to find similar items based on their MinHash signatures. The threshold parameter determines the similarity threshold for considering two items as similar. In this case, a threshold of 0.5 means that two items must have at least 50% similarity in their MinHash signatures to be considered similar.

entity_hashes = {}
for _, row in entity_names.iterrows():
    m = MinHash(num_perm=64)
    for g in row['ngrams']: m.update(g.encode())
    lsh.insert(row['id'], m)
    entity_hashes[row['id']] = m

candidates = []
for _, row in vessel_names.iterrows():
    m = MinHash(num_perm=64)
    for g in row['ngrams']: m.update(g.encode())
    matches = lsh.query(m)
    for match_id in matches:
        candidates.append({'vessel_id': row['id'], 'vessel_name': row['caption'],
                           'entity_id': match_id})


In [ ]:
entity_ngram_map = entity_names.set_index('id')['ngrams'].to_dict() # .set_index('id') sets the 'id' column as the index of the DataFrame, and ['ngrams'] selects the 'ngrams' column. 

# The .to_dict() method converts the selected column into a dictionary where the keys are the index values (entity IDs) and the values are the corresponding n-gram sets. This allows for quick lookup of n-grams for any given entity ID.

results = []
for c in candidates:
    A = entity_ngram_map.get(c['entity_id'], set())
    B = vessel_names.loc[vessel_names['id'] == c['vessel_id'], 'ngrams'].values
    if len(B) == 0: continue
    B = B[0]
    jaccard = len(A & B) / len(A | B) if A | B else 0
    if jaccard >= 0.5:
        results.append({**c, 'entity_name': entity_names.loc[entity_names['id']==c['entity_id'],'name'].values[0],
                        'jaccard': round(jaccard, 3)})

df_rq1 = pd.DataFrame(results).sort_values('jaccard', ascending=False)
df_rq1

# The 0.5 threshold is intentionally loose — vessel names after renaming rarely share more than half their trigrams. Review the output and raise it if there is too much noise.

# A jaccard of 1 indicates that the vessel name and entity name are identical in terms of their trigrams, while a jaccard of 0.5 indicates that they share half of their trigrams. The higher the jaccard value, the more similar the two names are.

### RQ2: Is there a statistically significant association between a vessel's country of sanctioned ownership and its registered flag state, in particular, do Russian- and Iranian-linked vessels disproportionately reflag to conduit jurisdictions?

A-Priori on `{country:XX, flag:YY}` baskets.

In [ ]:
conduit_jurisdictions = ['PA', 'VU', 'MT', 'KM', 'PW', 'TV']

# Here conduit_jurisdictions is a list of country codes that are considered to be conduit jurisdictions. These are countries or territories that are often used for routing financial transactions or business activities, sometimes to take advantage of favorable tax laws or regulatory environments.


In [ ]:
# To perform the A-Priori algorithm, we need to create a "basket" of items for each row in the maritime dataset. This basket will include the countries and flags associated with each vessel, as well as a flag indicating whether the vessel is registered in a conduit jurisdiction. 

# The A-Priori algorithm serves to identify frequent itemsets in these baskets, which can help uncover patterns and relationships between different attributes of the vessels.

def build_basket(row):
    items = []
    for c in str(row['countries']).split(';'): # for c in str(row['countries']).split(';') splits the 'countries' string into a list of individual country codes using the semicolon as a delimiter. This allows us to process each country code separately.
        if c.strip(): items.append(f"country:{c.strip()}") # c.strip removes any leading or trailing whitespace from the country code, and if the resulting string is not empty, it appends a formatted string "country:{country_code}" to the items list. This ensures that only valid country codes are included in the basket.
    if pd.notna(row['flag']): # pd.notna(row['flag']) checks if the 'flag' value is not NaN (i.e., it has a valid value). If the flag is present, it appends a formatted string "flag:{flag_code}" to the items list. Additionally, it checks if the flag is in the conduit_jurisdictions list and appends "conduit:True" to indicate that the vessel is registered in a conduit jurisdiction.
        items.append(f"flag:{row['flag'].strip()}") # appends a formatted string "flag:{flag_code}" to the items list, where {flag_code} is the cleaned-up flag value from the row. This ensures that the flag information is included in the basket for each vessel.
        if row['flag'].strip() in conduit_jurisdictions: 
            items.append("conduit:True")
    return list(set(items))

df_maritime['basket'] = df_maritime.apply(build_basket, axis=1)

In [ ]:
# A Transaction Encoder is used to convert the list of items in each basket into a one-hot encoded format, which is suitable for the A-Priori algorithm. Each unique item in the baskets will become a column in the resulting DataFrame, with binary values indicating the presence or absence of that item in each basket.

te = TransactionEncoder() 

# The function TransactionEncoder() from the mlxtend library is used to transform a list of transactions (in this case, the baskets of items) into a one-hot encoded format. This is necessary for applying the A-Priori algorithm, which requires the input data to be in a binary format where each column represents an item and each row represents a transaction.

# In our example, with sanctions data, each basket represents a vessel and contains items such as countries, flags, and conduit status. The TransactionEncoder will create a binary matrix where each unique item in the baskets becomes a column, and each row corresponds to a vessel. A value of 1 indicates the presence of an item in the basket, while a value of 0 indicates its absence.

te_array = te.fit_transform(df_maritime['basket'].tolist())

# df_maritime['basket'] is a pandas series containing the baskets of items for each vessel. The .tolist() method converts this series into a list of lists, where each inner list represents the items in a single basket.

# te.fit_transform takes the list of baskets and transforms it into a one-hot encoded array. Each row in the resulting array corresponds to a vessel, and each column corresponds to a unique item (country, flag, or conduit status). The values in the array are binary (1 or 0), indicating whether the item is present in the basket for that vessel.
 
basket_df = pd.DataFrame(te_array, columns=te.columns_)

freq_items = apriori(basket_df, min_support=0.03, use_colnames=True, max_len=3)

# apriori() is a function from the mlxtend library that implements the A-Priori algorithm for mining frequent itemsets from transactional data. In this context, it is used to identify combinations of countries, flags, and conduit status that frequently co-occur in the maritime dataset

# Support at 3% on the maritime dataset should produce a non-trivial rule set — adjust down to 1% if it returns nothing.

In [ ]:
rules = association_rules(freq_items, metric='confidence', min_threshold=0.4)
rules = rules[rules['lift'] > 1.2].sort_values('lift', ascending=False)

In [ ]:
# Focus on rules that conclude in a conduit flag
flag_rules = rules[rules['consequents'].apply(lambda x: any('flag:' in i or 'conduit:' in i for i in x))]
flag_rules[['antecedents','consequents','support','confidence','lift']]

The `conduit:True` item allows to isolate rules of the form `{country:RU} → {conduit:True}` directly, without having to enumerate each individual flag state.

### RQ3: Given an entity's current dataset memberships, which additional sanctions programmes is their profile most consistent with i.e., which programmes have they not yet been listed in but should be?

Binary entity × dataset matrix, factorised to find missing listings.

In [ ]:
# Parse the 'datasets' column (semicolon-separated)

# df_consolidated['dataset_list'] = df_consolidated['dataset'].str.split(';').apply(
#     lambda x: [d.strip() for d in x] if isinstance(x, list) else []
# )
# `isinstance(x, list)` is always True after `.str.split(';')` on non-null values. For null values, `str(NaN).split(';')` returns `['nan']`, so null entities get `'nan'` added to their dataset list. Fix: `df_consolidated['dataset'].fillna('').str.split(';').apply(lambda x: [d.strip() for d in x if d.strip()])` |

df_consolidated['dataset_list'] = df_consolidated['dataset'].fillna('').str.split(';').apply(
    lambda x: [d.strip() for d in x if d.strip()]
)

all_datasets = sorted({d for lst in df_consolidated['dataset_list'] for d in lst})
entity_ids   = df_consolidated['id'].tolist()

dataset_idx  = {d: i for i, d in enumerate(all_datasets)}
entity_idx   = {e: i for i, e in enumerate(entity_ids)}

M = np.zeros((len(entity_ids), len(all_datasets)), dtype=np.float32)
for _, row in df_consolidated.iterrows():
    i = entity_idx[row['id']]
    for d in row['dataset_list']:
        if d in dataset_idx:
            M[i, dataset_idx[d]] = 1.0
            
# RQ3 matrix factorization is trained on positive dataset memberships only, which can bias scores in a sparse binary matrix.

# Use an implicit-feedback-aware factorization approach, negative sampling, or a loss function designed for missing vs observed entries.

In [ ]:
np.random.seed(42)
k   = 10   # latent dimensions — keep small for speed
eta = 0.01
lam = 0.02

U = np.random.normal(0, 0.1, (len(entity_ids), k)).astype(np.float32)
V = np.random.normal(0, 0.1, (k, len(all_datasets))).astype(np.float32)

rows, cols = np.nonzero(M)
for epoch in range(5):
    perm = np.random.permutation(len(rows))
    for p in perm:
        i, j  = rows[p], cols[p]
        e     = M[i,j] - U[i] @ V[:,j]
        U[i] += eta * (e * V[:,j] - lam * U[i])
        V[:,j]+= eta * (e * U[i]  - lam * V[:,j])
    print(f"Epoch {epoch+1} done")
    
# Epoch is a single pass through the entire dataset during the training process. In this context, it refers to one complete iteration of updating the latent feature matrices U and V based on the observed entries in the matrix M. Each epoch allows the model to learn and adjust its parameters to better approximate the underlying patterns in the data. 

In [ ]:
scores = U @ V   # predicted relevance for every (entity, dataset) pair

# Mask already-known entries so we only look at gaps
scores[M == 1] = -np.inf

# Top-5 missing dataset recommendations per entity (sample: first 20 entities)
for i in range(20):
    top_datasets = np.argsort(scores[i])[::-1][:5]
    entity_name  = df_consolidated.iloc[i]['name']
    recs         = [all_datasets[j] for j in top_datasets]
    print(f"{entity_name}: {recs}")
    
# recs is a list of the top 5 recommended datasets for the entity at index i in the df_consolidated DataFrame. The recommendations are based on the predicted relevance scores calculated from the matrix factorization model (U @ V). The higher the score, the more relevant the dataset is considered for that entity.

### RQ4: Which non-sanctioned PEPs are most similar, by attribute profile, to the entities in df_sanctioned_people_peps, and should therefore be prioritised for enhanced due diligence?

Jaccard on attribute sets to rank unsanctioned PEPs by similarity to sanctioned ones.

In [ ]:
def attr_set_peps(row):
    # For df_peps_work (no suffix)
    countries = {f"c:{x.strip()}" for x in str(row.get('countries', '')).split(';') if x.strip()}
    return countries

def attr_set_sanctioned(row):
    # For df_sanctioned_people_peps (suffixed from merge)
    countries = {f"c:{x.strip()}" for x in str(row.get('countries_peps', '')).split(';') if x.strip()}
    return countries

sanctioned_sets = df_sanctioned_people_peps.apply(attr_set_sanctioned, axis=1).tolist()

non_sanctioned = df_peps_work[~df_peps_work['id'].isin(df_sanctioned_people_peps['id'])].copy()
non_sanctioned['attr_set'] = non_sanctioned.apply(attr_set_peps, axis=1)

# Sanity check — these must not be empty
print(sanctioned_sets[:3])
print(non_sanctioned['attr_set'].head(3))

In [ ]:
def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 0.0

# Mean Jaccard to the nearest 50 sanctioned PEPs (sample for speed)
sample_sanctioned = sanctioned_sets[:50]

# The RQ4 risk score is computed from a fixed first-50 sanctioned PEP sample rather than the closest neighbours.
# Use a true nearest-neighbour / full-sample similarity aggregation, or clearly label the result as an approximate sample-based score.

non_sanctioned['risk_score'] = non_sanctioned['attr_set'].apply(
    lambda s: np.mean([jaccard(s, ref) for ref in sample_sanctioned]).round(2)
)

# The sample of 50 sanctioned PEPs is a speed concession — computing against all 3,000+ sanctioned PEPs would be slow without vectorisation. If the results look thin, increase the sample. You can also replace the mean with the max similarity to favour PEPs that are very close to at least one sanctioned entity, rather than broadly similar to many.

In [ ]:
non_sanctioned.sort_values('risk_score', ascending=False)[['name','countries','risk_score']]

## Part III: Machine Learning and Predictive Modeling
---

Can we predict whether a PEP will also appear in the consolidated sanctions list using only the features available in their PEP profile (country count, alias count, identifier count, dataset count, top-country indicators)?

How much better does a high-dimensional SGD classifier perform over LDA/QDA when one-hot country features are added, and what does the gap reveal about the linearity assumption?

Binary classification — predict whether a PEP will also appear in the consolidated sanctions list.
- Label `1`: PEP ID is in `df_sanctioned_people_peps`
- Label `0`: PEP ID is in `df_peps_work` but not in `df_sanctioned_people_peps`

<!-- One paragraph. State the task, the label definition, and the three models in order of complexity: LDA (linear boundary, equal covariance) → QDA (quadratic boundary, per-class covariance) → SGD (high-dimensional, sparse one-hot features). Note that ML is exploratory here, not the primary contribution. -->


In [ ]:
df_part3 = df_peps_work.copy()

# Simple numeric features
df_part3['n_countries']   = df_peps_work['countries'].fillna('').str.split(';').apply(lambda x: len([i for i in x if i.strip()]))
df_part3['n_aliases']     = df_peps_work['aliases'].fillna('').str.split(';').apply(lambda x: len([i for i in x if i.strip()]))
df_part3['n_identifiers'] = df_peps_work['identifiers'].fillna('').str.split(';').apply(lambda x: len([i for i in x if i.strip()]))
df_part3['has_birthdate'] = df_peps_work['birth_date'].notna().astype(int)
df_part3['n_programs']    = df_peps_work['program_ids'].fillna('').str.split(';').apply(lambda x: len([i for i in x if i.strip()]))

In [ ]:
# Label
df_part3['label'] = df_part3['id'].isin(df_sanctioned_people_peps['id']).astype(int)

print(df_part3['label'].value_counts())

FEATURES_SIMPLE = ['n_aliases', 'has_birthdate', 'n_identifiers']
X_simple = df_part3[FEATURES_SIMPLE].fillna(0).values
y        = df_part3['label'].values

In [ ]:
# Check the class distribution before moving on. If the positive class is below 10% you have an imbalance problem — `class_weight='balanced'` fixes it for SGD, and `priors` for LDA/QDA.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_simple, y, test_size=0.2, stratify=y, random_state=42
)

# `stratify=y` is not optional here — without it, a skewed split will produce a test set with almost no positives and inflated accuracy.

In [ ]:
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)
y_pred_lda = lda.predict(X_test)

print("LDA")
print(metrics.classification_report(y_test, y_pred_lda))

In [ ]:
# PLOT LDA decision boundary

In [ ]:
qda = QuadraticDiscriminantAnalysis(reg_param=0.1) # # QDA may throw a `LinAlgError` or `collinearity` warning if features are nearly constant within a class — which is likely given that most PEPs have `n_programs = 0`. Add `reg_param=0.1` to the constructor if it fails.
qda.fit(X_train, y_train)
y_pred_qda = qda.predict(X_test)

print("QDA")
print(metrics.classification_report(y_test, y_pred_qda))

In [ ]:
# PLOT QDA decision boundary

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix

# hstack, csr_matrix are used to efficiently combine sparse matrices. In this context, they are used to concatenate the one-hot encoded country features with the simple numeric features for the SGD classifier.

# hstack is a function that horizontally stacks sparse matrices, allowing you to combine multiple feature sets into a single feature matrix. This is particularly useful when dealing with high-dimensional data, such as one-hot encoded categorical features.

# csr_matrix is a compressed sparse row matrix format that efficiently stores sparse data. It is used here to represent the one-hot encoded country features in a memory-efficient manner, as many of the entries will be zero.

# AUC stands for Area Under the Curve, specifically it is a metric used to evaluate the performance of binary classification models. It measures the ability of the model to distinguish between the positive and negative classes. A higher AUC value indicates better model performance, with a value of 1.0 representing perfect classification and a value of 0.5 indicating random guessing. 

# Interpretation of AUC values:
# 0.5: No discriminative power. The model behaves as if it were flipping a coin.
# 0.5 - 0.7: Poor or sufficient accuracy.
# 0.7 - 0.8: Acceptable performance.
# 0.8 - 0.9: Excellent performance.
# > 0.9: Exceptional / near-perfect performance.
# 1.0: Total and perfect separation between classes

# The AUC is computed via the library function `roc_auc_score`, which calculates the area under the receiver operating characteristic (ROC) curve. The ROC curve plots the true positive rate (sensitivity) against the false positive rate (1-specificity) at various threshold settings. The AUC value provides a single scalar value to summarize the model's performance across all classification thresholds. The function belongs to the library `sklearn.metrics`, which provides a range of tools for evaluating the performance of machine learning models. 

# One-hot encode top 30 countries
top_countries = (
    df_part3['countries'].fillna('').str.split(';').explode().str.strip()
    .value_counts().head(30).index.tolist()
)

def country_onehot(countries_str):
    present = set(countries_str.split(';')) # present means the set of countries that are present in the input string `countries_str`. The input string is split by semicolons to separate individual country codes, and then converted into a set to eliminate duplicates and allow for efficient membership testing.
    return [1 if c in present else 0 for c in top_countries] # top_countries is a list of the 30 most frequently occurring countries in the dataset. The function iterates over this list and checks if each country is present in the `present` set. If a country is found in the `present` set, it returns 1 (indicating presence), otherwise it returns 0 (indicating absence). This results in a binary vector representing the presence or absence of each of the top 30 countries for a given entity.

X_countries = np.array(df_part3['countries'].fillna('').apply(country_onehot).tolist())
X_sgd = np.hstack([X_simple, X_countries])

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_sgd, y, test_size=0.2, stratify=y, random_state=42
)

sgd = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SGDClassifier(loss='modified_huber', class_weight='balanced',
                          max_iter=100, random_state=42))
])

# Code reference: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html

sgd.fit(X_tr_s, y_tr_s)

print("SGD")
print(metrics.classification_report(y_te_s, sgd.predict(X_te_s)))
print(f"AUC: {roc_auc_score(y_te_s, sgd.predict_proba(X_te_s)[:,1]):.3f}")

In [ ]:
# Plot SGD decision boundary on the first two features
# Code Reference: https://scikit-learn.org/stable/auto_examples/linear_model/plot_sgd_iris.html

from sklearn.inspection import DecisionBoundaryDisplay

X_vis = X_te_s[:, :2]  # Use the first two features for visualization
y_vis = y_te_s
colors = "byr"

# Train a separate 2D SGD model for the decision boundary plot
sgd2 = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SGDClassifier(loss='modified_huber', class_weight='balanced', max_iter=100, random_state=42))
])
sgd2.fit(X_tr_s[:, :2], y_tr_s)

ax = plt.gca()
DecisionBoundaryDisplay.from_estimator(
    sgd2,
    X_vis,
    ax=ax,
    response_method="predict",
    xlabel=FEATURES_SIMPLE[0],
    ylabel=FEATURES_SIMPLE[1],
)

plt.axis("tight")

# Plot the validation points
for i, color in zip(sgd2.named_steps['clf'].classes_, colors):
    idx = (y_vis == i)
    plt.scatter(
        X_vis[idx, 0],
        X_vis[idx, 1],
        c=color,
        label=str(i),
        edgecolor="black",
        s=20,
    )

plt.title("SGD Classifier Decision Boundary")
plt.axis("tight")
plt.legend()
plt.show()

The SGD decision boundary in the (n_aliases, has_birthdate) feature plane reveals a coherent risk logic: entities lacking both a recorded birth date and alias documentation are classified as non-sanctioned, while the presence of either feature — particularly a birth date — strongly predicts sanctions designation. This is consistent with the feature mean analysis: sanctioned PEPs are systematically better documented, reflecting the identity-verification requirements of the designation process rather than any inherent behavioural difference.

In [ ]:
print(df_part3.groupby('label')[FEATURES_SIMPLE].mean().round(3))

In [ ]:
# One Hot Encoder 
# One-hot encoding is a method to convert categorical variables into a binary matrix representation. Each unique category value is transformed into a new binary column (feature), where a value of 1 indicates the presence of that category and 0 indicates its absence. This is particularly useful for machine learning algorithms that require numerical input, as it allows categorical data to be represented in a format that can be processed effectively.

# from sklearn.preprocessing import OneHotEncoder

# df = pd.DataFrame({'Colore': ['Rosso', 'Blu', 'Verde', 'Blu']})

# # Inizializziamo l'encoder
# encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# # Trasformiamo la colonna
# encoded_array = encoder.fit_transform(df[['Colore']])

# # Creiamo un nuovo DataFrame con i nomi delle colonne corretti
# encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(['Colore']))

# # Opzionale: se si vuole convertire gli 0 e 1 in interi anziché float
# encoded_df = encoded_df.astype(int)

# # Uniamo al dataframe originale (o sostituiamo la colonna)
# df_final = pd.concat([df, encoded_df], axis=1).drop('Colore', axis=1)
# print(df_final)


In [ ]:
results = pd.DataFrame([
    {'Model': 'LDA',  'Features': 'simple (3)',       'AUC': roc_auc_score(y_test, lda.predict_proba(X_test)[:,1])},
    {'Model': 'QDA',  'Features': 'simple (3)',       'AUC': roc_auc_score(y_test, qda.predict_proba(X_test)[:,1])},
    {'Model': 'SGD',  'Features': 'simple + one-hot (35)', 'AUC': roc_auc_score(y_te_s, sgd.predict_proba(X_te_s)[:,1])},
]).round(3)
results

In [ ]:
# **The comparison is confounded.** LDA/QDA use 5 features; SGD uses 35. If SGD outperforms, you cannot tell whether that is because of the model or because of the additional features. If you want a clean comparison, also run SGD on the 5 simple features. One extra fit call.

# **LDA's Gaussian assumption is violated.** Count features like `n_aliases` and `n_programs` are zero-inflated and right-skewed — the opposite of normally distributed. LDA will still run and produce a result, but the theoretical guarantees do not hold. Acknowledge this in the methodology markdown as a known limitation, not something to fix — it is acceptable scope given ML is not the course focus.

## Conclusions
---